In [ ]:
import sys; sys.path.insert(0, '..')


# Empirical Validation of CFAD

This notebook reproduces the empirical validation outputs used in the paper: detection delay, ROC behavior, and sensitivity to window size.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import cfad
from cfad.utils import simulate_levy_returns

try:
    from joblib import Parallel, delayed
    JOBLIB_AVAILABLE = True
except ImportError:
    JOBLIB_AVAILABLE = False

np.random.seed(2026)

def run_parallel(func, iterable, n_jobs=-1):
    items = list(iterable)
    if JOBLIB_AVAILABLE:
        try:
            return Parallel(n_jobs=n_jobs, prefer='threads')(
                delayed(func)(item) for item in items
            )
        except Exception as exc:
            print(f'Parallel fallback to n_jobs=1 due to: {exc}')
            return Parallel(n_jobs=1, prefer='threads')(
                delayed(func)(item) for item in items
            )
    return [func(item) for item in items]

def cusum_alarm_indices(scores, mu0, sigma0, h=4.0, k=0.5):
    scores = np.asarray(scores, dtype=np.float64)
    sigma0 = float(sigma0) + 1e-12
    slack = k * sigma0
    s_pos, s_neg = 0.0, 0.0
    alarms = []
    for t, val in enumerate(scores):
        z = (val - mu0) / sigma0
        s_pos = max(0.0, s_pos + z - slack)
        s_neg = max(0.0, s_neg - z - slack)
        if s_pos > h or s_neg > h:
            alarms.append(t)
            s_pos, s_neg = 0.0, 0.0
    return np.asarray(alarms, dtype=np.int64)

def positive_in_tail(alarm_indices, n_obs, tail=60):
    alarms = np.asarray(alarm_indices, dtype=np.int64)
    if alarms.size == 0:
        return False
    cutoff = max(0, int(n_obs) - int(tail))
    return bool(np.any(alarms >= cutoff))

def auc_from_rates(fpr, tpr):
    fpr = np.asarray(fpr, dtype=np.float64)
    tpr = np.asarray(tpr, dtype=np.float64)
    order = np.argsort(fpr)
    return float(np.trapezoid(tpr[order], fpr[order]))

def make_jump_series(seed, t=500):
    rng = np.random.default_rng(seed)
    left = rng.normal(0.0, 0.01, 300)
    right = simulate_levy_returns(200, alpha=1.5, scale=0.012, seed=10_000 + seed)
    return np.concatenate([left, right])

def make_null_series(seed, t=400):
    rng = np.random.default_rng(seed)
    return rng.normal(0.0, 0.01, t)

def make_alt_series(seed, t=400):
    rng = np.random.default_rng(seed)
    left = rng.normal(0.0, 0.01, 300)
    right = simulate_levy_returns(100, alpha=1.5, scale=0.012, seed=20_000 + seed)
    return np.concatenate([left, right])

def baseline_state(series, calibration_frac=0.4):
    sq = np.asarray(series, dtype=np.float64) ** 2
    n_cal = max(10, int(calibration_frac * len(sq)))
    mu0 = float(np.mean(sq[:n_cal]))
    sigma0 = float(np.std(sq[:n_cal], ddof=1)) + 1e-12
    return sq, mu0, sigma0

def cfad_state(series, window=60, step=5, calibration_frac=0.4):
    report = cfad.detect(
        np.asarray(series, dtype=np.float64),
        window=window,
        step=step,
        calibration_frac=calibration_frac,
        h=4.0,
    )
    return report

print('joblib available:', JOBLIB_AVAILABLE)


## 1. Detection delay distribution

We simulate 300 jump-regime series of length $T=500$, with the break at sample index 300. The detector is run with `window=60`, `step=5`, `calibration_frac=0.4`, `h=4.0`.

The detection delay is measured as

$$\text{delay} = \frac{t_{\text{first alarm}} - 300}{5},$$

so delays are expressed in window units (5 samples per step).


In [ ]:
N_DELAY = 300
CHANGE_POINT = 300
STEP = 5

def run_one_delay(seed):
    series = make_jump_series(seed)

    report = cfad_state(series, window=60, step=STEP, calibration_frac=0.4)
    if len(report.alarm_indices) == 0:
        cfad_first_sample = np.nan
    else:
        first_alarm_window = int(report.alarm_indices[0])
        cfad_first_sample = float(report.window_end_indices[first_alarm_window] - 1)

    sq, b_mu, b_sigma = baseline_state(series, calibration_frac=0.4)
    b_alarms = cusum_alarm_indices(sq, b_mu, b_sigma, h=4.0, k=0.5)
    baseline_first_sample = float(b_alarms[0]) if b_alarms.size > 0 else np.nan

    return cfad_first_sample, baseline_first_sample

delay_results = run_parallel(run_one_delay, range(N_DELAY), n_jobs=-1)
delay_results = np.asarray(delay_results, dtype=np.float64)

cfad_first_sample = delay_results[:, 0]
baseline_first_sample = delay_results[:, 1]

cfad_delay_windows = (cfad_first_sample - CHANGE_POINT) / STEP
baseline_delay_windows = (baseline_first_sample - CHANGE_POINT) / STEP

median_cfad_delay = float(np.nanmedian(cfad_delay_windows))

fig_delay, ax_delay = plt.subplots(figsize=(8, 5))
valid_delay = cfad_delay_windows[np.isfinite(cfad_delay_windows)]
ax_delay.hist(valid_delay, bins=30, color='tab:blue', edgecolor='white', alpha=0.85)
ax_delay.axvline(median_cfad_delay, color='black', linestyle='--', linewidth=1.2, label='Median')
ax_delay.set_xlabel('Detection delay (windows)')
ax_delay.set_ylabel('Frequency')
ax_delay.set_title(f'Detection Delay Distribution (median = {median_cfad_delay:.2f} windows)')
ax_delay.legend(loc='upper right')
ax_delay.grid(alpha=0.25)
fig_delay.tight_layout()

fig_dir = Path('../paper/figures')
fig_dir.mkdir(parents=True, exist_ok=True)
delay_path = fig_dir / 'detection_delay.png'
fig_delay.savefig(delay_path, dpi=150, bbox_inches='tight')
print('Saved:', delay_path.resolve())
print('Valid detections:', valid_delay.size, 'of', N_DELAY)
fig_delay


## 2. ROC curve comparison

We compare two detectors on identical null/alternative simulations:

1. **CFAD**: contour residue score + CUSUM
2. **Baseline**: CUSUM on squared returns $r_t^2$

If `benchmarks/roc_auc.txt` exists, we load it as an external reference AUC from the benchmark script.


In [ ]:
N_SIM_ROC = 100
H_GRID = np.linspace(1.0, 10.0, 30)
TAIL = 60

bench_auc_path = Path('../benchmarks/roc_auc.txt')
if bench_auc_path.exists():
    benchmark_auc = float(bench_auc_path.read_text(encoding='utf-8').strip())
    print(f'Loaded benchmark AUC from file: {benchmark_auc:.3f}')
else:
    benchmark_auc = np.nan
    print('No benchmark AUC file found; using notebook recomputation.')

null_series = [make_null_series(seed) for seed in range(N_SIM_ROC)]
alt_series = [make_alt_series(50_000 + seed) for seed in range(N_SIM_ROC)]

def compute_states(series):
    report = cfad_state(series, window=60, step=5, calibration_frac=0.4)
    sq, b_mu, b_sigma = baseline_state(series, calibration_frac=0.4)
    return (
        report.scores,
        float(report.mu0),
        float(report.sigma0),
        sq,
        float(b_mu),
        float(b_sigma),
    )

null_states = run_parallel(compute_states, null_series, n_jobs=-1)
alt_states = run_parallel(compute_states, alt_series, n_jobs=-1)

def positives_for_h(states, h_value):
    cfad_pos = []
    base_pos = []
    for c_scores, c_mu, c_sigma, b_scores, b_mu, b_sigma in states:
        c_alarms = cusum_alarm_indices(c_scores, c_mu, c_sigma, h=h_value, k=0.5)
        b_alarms = cusum_alarm_indices(b_scores, b_mu, b_sigma, h=h_value, k=0.5)
        cfad_pos.append(positive_in_tail(c_alarms, len(c_scores), tail=TAIL))
        base_pos.append(positive_in_tail(b_alarms, len(b_scores), tail=TAIL))
    return np.asarray(cfad_pos, dtype=bool), np.asarray(base_pos, dtype=bool)

cfad_fpr = []
cfad_tpr = []
base_fpr = []
base_tpr = []

for h_val in H_GRID:
    c_null, b_null = positives_for_h(null_states, float(h_val))
    c_alt, b_alt = positives_for_h(alt_states, float(h_val))
    cfad_fpr.append(float(np.mean(c_null)))
    cfad_tpr.append(float(np.mean(c_alt)))
    base_fpr.append(float(np.mean(b_null)))
    base_tpr.append(float(np.mean(b_alt)))

cfad_fpr = np.asarray(cfad_fpr, dtype=np.float64)
cfad_tpr = np.asarray(cfad_tpr, dtype=np.float64)
base_fpr = np.asarray(base_fpr, dtype=np.float64)
base_tpr = np.asarray(base_tpr, dtype=np.float64)

cfad_auc = auc_from_rates(cfad_fpr, cfad_tpr)
baseline_auc = auc_from_rates(base_fpr, base_tpr)

c_null_h4, b_null_h4 = positives_for_h(null_states, 4.0)
c_alt_h4, b_alt_h4 = positives_for_h(alt_states, 4.0)
cfad_fpr_h4 = float(np.mean(c_null_h4))
baseline_fpr_h4 = float(np.mean(b_null_h4))

print(f'Notebook CFAD AUC: {cfad_auc:.3f}')
print(f'Notebook baseline AUC: {baseline_auc:.3f}')

# CFAD-only ROC figure (for full validation figure set).
fig_roc, ax_roc = plt.subplots(figsize=(6.5, 5))
order_c = np.argsort(cfad_fpr)
ax_roc.plot(cfad_fpr[order_c], cfad_tpr[order_c], marker='o', linewidth=1.5, label=f'CFAD AUC={cfad_auc:.3f}')
ax_roc.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1.0)
if np.isfinite(benchmark_auc):
    ax_roc.text(0.05, 0.92, f'benchmark AUC file: {benchmark_auc:.3f}', transform=ax_roc.transAxes, fontsize=9)
ax_roc.set_xlabel('FPR')
ax_roc.set_ylabel('TPR')
ax_roc.set_title('CFAD ROC Curve')
ax_roc.grid(alpha=0.3)
ax_roc.legend(loc='lower right')
fig_roc.tight_layout()

roc_path = fig_dir / 'roc_curve.png'
fig_roc.savefig(roc_path, dpi=150, bbox_inches='tight')
print('Saved:', roc_path.resolve())

# ROC comparison figure.
fig_cmp, ax_cmp = plt.subplots(figsize=(7, 5.5))
order_b = np.argsort(base_fpr)
ax_cmp.plot(cfad_fpr[order_c], cfad_tpr[order_c], marker='o', linewidth=1.8, label=f'CFAD (AUC={cfad_auc:.3f})')
ax_cmp.plot(base_fpr[order_b], base_tpr[order_b], marker='s', linewidth=1.8, label=f'Baseline r^2-CUSUM (AUC={baseline_auc:.3f})')
ax_cmp.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1.0)
ax_cmp.set_xlabel('False Positive Rate')
ax_cmp.set_ylabel('True Positive Rate')
ax_cmp.set_title('ROC Comparison: CFAD vs Moment-Based Baseline')
ax_cmp.grid(alpha=0.3)
ax_cmp.legend(loc='lower right')
fig_cmp.tight_layout()

roc_cmp_path = fig_dir / 'roc_comparison.png'
fig_cmp.savefig(roc_cmp_path, dpi=150, bbox_inches='tight')
print('Saved:', roc_cmp_path.resolve())
fig_cmp


## 3. Sensitivity to window size

We evaluate CFAD AUC at window sizes $[30, 60, 90, 120]$ using 100 null and 100 jump-regime replicates. Error bars are bootstrap standard deviations from 10 resamples.


In [ ]:
WINDOWS = [30, 60, 90, 120]
N_BOOT = 10
boot_rng = np.random.default_rng(77)

def compute_cfad_only_state(args):
    series, window = args
    report = cfad_state(series, window=window, step=5, calibration_frac=0.4)
    return report.scores, float(report.mu0), float(report.sigma0)

def cfad_positive_matrix(states, h_grid, tail=60):
    pos = np.zeros((len(h_grid), len(states)), dtype=bool)
    for i, h_val in enumerate(h_grid):
        for j, (scores, mu0, sigma0) in enumerate(states):
            alarms = cusum_alarm_indices(scores, mu0, sigma0, h=float(h_val), k=0.5)
            pos[i, j] = positive_in_tail(alarms, len(scores), tail=tail)
    return pos

window_auc = []
window_auc_std = []

for w in WINDOWS:
    null_states_w = run_parallel(compute_cfad_only_state, [(s, w) for s in null_series], n_jobs=-1)
    alt_states_w = run_parallel(compute_cfad_only_state, [(s, w) for s in alt_series], n_jobs=-1)

    pos_null = cfad_positive_matrix(null_states_w, H_GRID, tail=TAIL)
    pos_alt = cfad_positive_matrix(alt_states_w, H_GRID, tail=TAIL)

    fpr_w = pos_null.mean(axis=1)
    tpr_w = pos_alt.mean(axis=1)
    auc_w = auc_from_rates(fpr_w, tpr_w)

    auc_boot = []
    n_sim = pos_null.shape[1]
    for _ in range(N_BOOT):
        idx_null = boot_rng.integers(0, n_sim, n_sim)
        idx_alt = boot_rng.integers(0, n_sim, n_sim)
        fpr_b = pos_null[:, idx_null].mean(axis=1)
        tpr_b = pos_alt[:, idx_alt].mean(axis=1)
        auc_boot.append(auc_from_rates(fpr_b, tpr_b))

    auc_std = float(np.std(np.asarray(auc_boot, dtype=np.float64), ddof=1))
    window_auc.append(float(auc_w))
    window_auc_std.append(auc_std)
    print(f'window={w:3d} -> AUC={auc_w:.3f} +/- {auc_std:.3f}')

window_auc = np.asarray(window_auc, dtype=np.float64)
window_auc_std = np.asarray(window_auc_std, dtype=np.float64)

fig_ws, ax_ws = plt.subplots(figsize=(7, 5))
ax_ws.errorbar(WINDOWS, window_auc, yerr=window_auc_std, fmt='o-', capsize=5, linewidth=1.8, color='tab:blue')
ax_ws.set_xlabel('Window size')
ax_ws.set_ylabel('AUC')
ax_ws.set_title('CFAD Sensitivity to Window Size')
ax_ws.grid(alpha=0.3)
fig_ws.tight_layout()

ws_path = fig_dir / 'window_sensitivity.png'
fig_ws.savefig(ws_path, dpi=150, bbox_inches='tight')
print('Saved:', ws_path.resolve())
fig_ws


## 4. Summary table


In [ ]:
median_cfad = float(np.nanmedian(cfad_delay_windows))
median_base = float(np.nanmedian(baseline_delay_windows))

table_lines = [
    '| Metric                   | cfad (contour) | Baseline (moments) |',
    '|--------------------------|----------------|--------------------|',
    f'| AUC                      | {cfad_auc:.3f}          | {baseline_auc:.3f}              |',
    f'| Median detection delay   | {median_cfad:.0f} windows     | {median_base:.0f} windows         |',
    f'| FPR at h=4.0             | {100*cfad_fpr_h4:.2f}%         | {100*baseline_fpr_h4:.2f}%            |',
]

print('\n'.join(table_lines))


## 5. Save all figures


In [ ]:
expected = [
    fig_dir / 'detection_delay.png',
    fig_dir / 'roc_curve.png',
    fig_dir / 'roc_comparison.png',
    fig_dir / 'window_sensitivity.png',
]

for p in expected:
    print(f'{p.name}:', 'OK' if p.exists() else 'MISSING')

all_exists = all(p.exists() for p in expected)
print('All required figures present:', all_exists)
